# obj1a · *Saccharomyces cerevisiae* (yeast-GEM)

| | Pergunta |
|---|---|
| T1 | O que há no ficheiro? |
| T2 | Cresce no meio default? |
| T3 | Crescimento com todas as trocas abertas? |
| T4 | Com tudo fechado, produz biomassa ou ATP do nada? |
| T5 | Que reações não fecham em massa? |
| T6 | A biomassa: escala, custo em ATP, componentes |
| T7 | De que compostos precisa mesmo? (meio mínimo) |
| T8 | Quais são indispensáveis? (fechar um de cada vez) |
| T9 | Cresce em aerobiose e em anaerobiose? |
| T11 | Se não cresce sem O₂, qual é o componente que bloqueia? |
| T12 | Há termos residuais da biomassa a decidir o fenótipo? |
| T13 | Que produtos secreta, em **intervalo** e não em ponto? |
| T14 | O carbono fecha? Qual o rendimento em etanol? |

Todas as funções estão em `scripts/testes.py`. Nenhuma altera o modelo. A curadoria está em `scripts/curadoria_minima.py`.

## 1. Preparação

In [1]:
import sys, warnings, os
from pathlib import Path
import cobra, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 80)

def raiz(p=None):
    p = (Path(p) if p else Path.cwd()).resolve()
    for c in (p, *p.parents):
        if (c / "models").is_dir() and (c / "scripts").is_dir():
            return c
    raise RuntimeError("raiz do projecto nao encontrada")

ROOT = raiz()
sys.path.insert(0, str(ROOT / "scripts"))
import testes as T
import mapeamento as MP

OUT = ROOT / "results" / "obj1a"
OUT.mkdir(parents=True, exist_ok=True)
print("raiz:", ROOT, "| cobrapy", cobra.__version__)

_lic = os.path.expanduser("~/gurobi.lic")
if os.path.exists(_lic):
    os.environ["GRB_LICENSE_FILE"] = _lic

ALIAS   = "Scer_yeastGEM"
MAPA    = MP.CROSS_MAP[ALIAS]
XML     = ROOT / "models" / "yeast-GEM.xml"

OBJECTIVO = MAPA["biomass"]     # r_2111, a reaccao que se maximiza
BIOMASSA  = "r_4041"            # r_4041, a que carrega a composicao
NGAM      = MAPA["ngam"]
O2        = MAPA["entradas"]["oxygen"]
ACUCARES  = [MAPA["entradas"]["glucose"], MAPA["entradas"]["fructose"]]
PRODUTOS  = {k: MAPA["saidas"][k] for k in
             ("ethanol", "glycerol", "acetate", "succinate", "acetaldehyde")
             if k in MAPA["saidas"]}

model = cobra.io.read_sbml_model(str(XML))
print(f"{model.id}: {len(model.reactions)} reaccoes")

raiz: /Users/Sara_1/Desktop/WORK/TASK3.1 | cobrapy 0.31.1
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
yeastGEM_v9__46__1__46__0: 4102 reaccoes


In [ ]:
# Registo dos achados. Cada linha é um defeito medido, com o teste que o mostrou, para posterior edição de curadoria.
ACHADOS = []

def achado(teste, o_que, gravidade="rever"):
    ACHADOS.append({"teste": teste, "achado": o_que, "gravidade": gravidade})
    print(f"  [{teste}] {o_que}")

## 2. T1 - o que há no ficheiro

Contagens, compartimentos e duas colunas: metabolitos sem fórmula (sem fórmula não há balanço de massa que se possa avaliar) e reações sem GPR (não permite interpretação ao nível genético).

In [3]:
print(pd.Series(T.t1_contar(model)).to_string())

modelo                                            yeastGEM_v9__46__1__46__0
reaccoes                                                               4102
metabolitos                                                            2748
genes                                                                  1143
compartimentos             c, ce, e, er, erm, g, gm, lp, m, mm, n, p, v, vm
trocas                                                                  270
metabolitos_sem_formula                                                  16
reaccoes_sem_GPR                                                       1420


## 3. T2 e T3 

O µ com tudo aberto não é uma taxa de crescimento: sem tectos de captação o modelo consome o que quiser e o número não tem significado biológico. Serve só para averiguar se o medium é restritivo ou se o problema é do modelo.

In [ ]:
r2 = T.t2_crescimento(model)
print(f"mu no meio do ficheiro: {r2['mu_meio_do_ficheiro']:.4f} h-1")
print(f"trocas abertas: {r2['trocas_abertas_a_captacao']}")
for rid in r2["ids"]:
    met = next(iter(model.reactions.get_by_id(rid).metabolites))
    print(f"   {rid:10s} {met.name}")

mu_aberto = T.t3_tudo_aberto(model)
print(f"\nmu com tudo aberto: {mu_aberto:.4f} h-1")

if r2["mu_meio_do_ficheiro"] < 1e-6 and mu_aberto > 1e-6:
    achado("T2/T3", "nao cresce no meio do ficheiro mas cresce com tudo aberto: "
                    "o meio do ficheiro esta restrito, nao ha defeito estrutural")
elif mu_aberto < 1e-6:
    achado("T3", "nao cresce nem com tudo aberto: defeito estrutural", "grave")

mu no meio do ficheiro : 0.0809 h-1
trocas abertas          : 16
   r_1654     ammonium
   r_1714     D-glucose
   r_1832     H+
   r_1861     iron(2+)
   r_1992     oxygen
   r_2005     phosphate
   r_2020     potassium
   r_2049     sodium
   r_2060     sulphate
   r_2100     H2O
   r_4593     chloride
   r_4594     Cu2(+)
   r_4595     Mn(2+)
   r_4596     Zn(2+)
   r_4597     Mg(2+)
   r_4600     Ca(2+)

mu com tudo aberto      : 59.4159 h-1  (nao e uma taxa)


## 4. T4 - sistema fechado

Com todas as fronteiras fechadas o modelo deve dar µ = 0 e ATP = 0. Qualquer valor positivo significa que fabrica matéria ou energia do nada, e que **todos** os µ deste modelo estão inflacionados.

In [ ]:
r4 = T.t4_sistema_fechado(model, NGAM)
print(pd.Series(r4).to_string())

if r4.get("biomassa", 0) > 1e-6:
    achado("T4", "cresce com tudo fechado: fuga de massa", "grave")
if r4.get("ATP", 0) > 1e-6:
    achado("T4", f"produz {r4['ATP_do_nada']:.3f} ATP com tudo fechado: "
                 f"ciclo gerador de energia", "grave")

biomassa_do_nada    0.0
ATP_do_nada         0.0


## 5. T5 — balanços de massa

Ignora-se o desvio de carga: o que permite fabricar matéria é o desequilíbrio
elementar. Duas colunas separam o que é erro do que é falta de informação: uma
reação com um metabolito sem fórmula aparece desequilibrada sem o estar.

As que atravessam a membrana plasmática são as graves, porque aí o desequilíbrio
fabrica matéria à custa do meio.

In [7]:
d5 = T.t5_balancos(model)
reais = d5[~d5.sem_formula] if len(d5) else d5
print(f"desequilibradas          : {len(d5)} de {len(model.reactions)}")
print(f"  por falta de formula   : {int(d5.sem_formula.sum()) if len(d5) else 0}")
print(f"  desequilibrio real     : {len(reais)}")
print(f"  tocam o extracelular   : {int(reais.toca_extracelular.sum()) if len(reais) else 0}")

if len(reais) and reais.toca_extracelular.any():
    print()
    print(reais[reais.toca_extracelular][["rxn","nome","compartimentos","desvio"]]
          .to_string(index=False))
    achado("T5", f"{int(reais.toca_extracelular.sum())} reaccoes desequilibradas "
                 f"atravessam a membrana plasmatica", "grave")
if len(d5):
    d5.to_csv(OUT / "T5_balancos.csv", index=False)

desequilibradas          : 203 de 4102
  por falta de formula   : 19
  desequilibrio real     : 184
  tocam o extracelular   : 0


## 6. T6 - a biomassa

A de biomassa dá-nos informação sobre a composição da célula. Fixa duas coisas de que tudo depende: a escala de µ (uma unidade de fluxo tem de valer 1 g, senão µ não está em h⁻¹) e o custo energético de crescer (o GAM).

O yeast-GEM divide a biomassa em sete pseudo-componentes (`protein`, `lipid`,`carbohydrate`, `RNA`, `DNA`, `cofactor`, `ion`), cada um já normalizado à sua fração da célula. Nesses modelos a escala de 1 g é imposta por construção e não se verifica pela soma das fórmulas - a coluna `escala` diz qual é o caso.

In [8]:
r6, f6 = T.t6_biomassa(model, BIOMASSA)
print(pd.Series(r6).to_string())

so_import = f6[f6.so_por_importacao] if len(f6) else f6
print(f"\ncomponentes folha: {len(f6)} | sem nenhuma sintese no modelo: {len(so_import)}")
if len(so_import):
    print(so_import[["metabolito","nome","formula","vem_de","tem_troca"]]
          .to_string(index=False))
    achado("T6", f"{len(so_import)} componentes da biomassa so entram por importacao: "
                 f"{', '.join(so_import.nome.head(6))}")
f6.to_csv(OUT / "T6_biomassa.csv", index=False)

reaccao                                                                                                     r_4041
n_reagentes                                                                                                     10
massa_por_formula_g                                                                                         2.2242
componentes_sem_massa                                                                                            7
soma_coef_componentes_sem_massa                                                                                7.0
soma_coef_1g                                                                                                   0.0
GAM_mmol_ATP_por_gDW                                                                                          55.3
escala                             imposta por construcao: 7 pseudo-componentes sem massa calculavel. Verificar...
desvio_de_1g_pct                                                                

## 7. T7 - meio mínimo

De que é que o modelo precisa mesmo para crescer. Parte-se de todas as trocas abertas e procura-se o conjunto mínimo que sustenta 10 % do µ máximo.

**Com o açúcar fixo.** Sem restrição, escolhe-se a fonte de carbono mais densa, porque assim minimiza o fluxo total de importação. Fixando a glucose e a frutose e fechando as outras fontes de carbono, a pergunta passa a ser: *com o açúcar do mosto, de que mais é que este modelo precisa?*

In [9]:
meio, mu_max = T.t7_meio_minimo(model, fraccao=0.1, fontes_de_C=ACUCARES, q_fonte=10.0)
print(f"mu maximo com o acucar a 10 mmol/gDW/h: {mu_max:.4f} h-1\n")
if meio is None:
    achado("T7", "nao cresce com glucose e frutose como unicas fontes de carbono", "grave")
else:
    tab7 = T.descrever_meio(model, meio)
    print(f"meio minimo: {len(tab7)} trocas")
    print(tab7.to_string(index=False))
    tab7.to_csv(OUT / "T7_meio_minimo.csv", index=False)

mu maximo com o acucar a 10 mmol/gDW/h: 1.6779 h-1

meio minimo: 14 trocas
 troca   metabolito formula  captacao
r_1992       oxygen      O2    4.6494
r_1709   D-fructose C6H12O6    2.0328
r_1654     ammonium     H4N    1.1045
r_2060     sulphate     O4S    0.0234
r_4528 triphosphate   O10P3    0.0156
r_2049       sodium      Na    0.0007
r_2020    potassium       K    0.0006
r_4595       Mn(2+)      Mn    0.0005
r_4593     chloride      Cl    0.0002
r_4597       Mg(2+)      Mg    0.0002
r_4594       Cu2(+)      Cu    0.0001
r_4596       Zn(2+)      Zn    0.0001
r_1861     iron(2+)      Fe    0.0000
r_4600       Ca(2+)      Ca    0.0000


## 8. Medium declarado

Os testes T8 e T9 precisam de um meio. Em vez do meio mínimo, utiliza-se um meio explícito: açúcar, azoto, minerais, vitaminas e os suplementos da auxotrofia anaeróbia.

**Os suplementos anaeróbios.** Em anaerobiose a *S. cerevisiae* não sintetiza esteróis nem ácidos gordos insaturados, porque as duas vias precisam de O₂. A lista abaixo é a do protocolo do próprio yeast-GEM (`code/otherChanges/anaerobicModel.m`, SysBioChalmers/yeast-GEM): ergosterol,lanosterol, zimosterol, 14-demetil-lanosterol, palmitoleato e oleato.

In [ ]:
# Suplementos da auxotrofia anaerobia — yeast-GEM anaerobicModel.m
ANAEROBIO = {
    "r_1757": "ergosterol",
    "r_1915": "lanosterol",
    "r_2106": "zimosterol",
    "r_2134": "14-demetil-lanosterol",
    "r_1994": "palmitoleato",
    "r_2189": "oleato",
}

Q_ACUCAR, Q_N, Q_VITAMINA, Q_SUPLEMENTO = 10.0, 5.0, 0.05, 0.05

def meio_de_trabalho(m, q_o2=0.0):
    """Fecha toda a captacao e abre um meio declarado. Altera in place."""
    T.fechar_captacao(m)
    T.abrir(m, ACUCARES, Q_ACUCAR)
    T.abrir(m, [MAPA["entradas"]["ammonium"]], Q_N)
    T.abrir(m, [MAPA["entradas"][k] for k in MP.INORGANICOS
                if k in MAPA["entradas"]], 1000.0)          # nao trazem carbono
    T.abrir(m, [MAPA["entradas"][k] for k in ("water", "proton", "co2")
                if k in MAPA["entradas"]], 1000.0)
    T.abrir(m, [MAPA["entradas"][k] for k in MP.VITAMINAS
                if k in MAPA["entradas"]], Q_VITAMINA)      # tem carbono: com tecto
    T.abrir(m, list(ANAEROBIO), Q_SUPLEMENTO)               # idem
    T.abrir(m, [MAPA["entradas"]["oxygen"]], q_o2)
    return m

with model:
    meio_de_trabalho(model, q_o2=0.0)
    print("trocas abertas no meio:")
    for r in sorted((r for r in model.exchanges if r.lower_bound < 0), key=lambda r: r.id):
        met = next(iter(r.metabolites))
        print(f"   {r.id:10s} {str(met.name)[:34]:36s} lb={r.lower_bound:8g}")

trocas abertas no meio de trabalho:
   r_1548     (R)-pantothenate                     lb=   -0.05
   r_1654     ammonium                             lb=      -5
   r_1671     biotin                               lb=   -0.05
   r_1672     carbon dioxide                       lb=   -1000
   r_1709     D-fructose                           lb=     -10
   r_1714     D-glucose                            lb=     -10
   r_1757     ergosterol                           lb=   -0.05
   r_1832     H+                                   lb=   -1000
   r_1861     iron(2+)                             lb=   -1000
   r_1915     lanosterol                           lb=   -0.05
   r_1947     myo-inositol                         lb=   -0.05
   r_1967     nicotinate                           lb=   -0.05
   r_1994     palmitoleate                         lb=   -0.05
   r_2005     phosphate                            lb=   -1000
   r_2020     potassium                            lb=   -1000
   r_2028     pyrid

## 9. T8 e T9 - compostos indispensáveis + oxigénio

**T8** fecha uma troca do meio de cada vez e mede o que sobra do crescimento. Um composto cuja remoção leva µ a zero é indispensável; os outros só limitam.

**T9** varia o oxigénio disponível no mesmo meio. As secreções vêm de pFBA: com FBA simples são um entre muitos ótimos e mudam entre execuções sem nada ter mudado.

In [11]:
with model:
    meio_de_trabalho(model, q_o2=2.0)
    d8 = T.t8_essenciais(model)

ess = d8[d8.essencial]
print(f"indispensaveis: {len(ess)} de {len(d8)} trocas abertas\n")
print(d8.to_string(index=False))
d8.to_csv(OUT / "T8_essenciais.csv", index=False)
if len(ess):
    achado("T8", f"indispensaveis no meio de trabalho: {', '.join(ess.metabolito)}",
           "informativo")

indispensaveis: 12 de 30 trocas abertas

 troca            metabolito  mu_sem_ele  perda_pct  essencial
r_1861              iron(2+)    0.000000      100.0       True
r_1992                oxygen    0.000000      100.0       True
r_2005             phosphate    0.000000      100.0       True
r_2020             potassium    0.000000      100.0       True
r_2049                sodium    0.000000      100.0       True
r_2060              sulphate    0.000000      100.0       True
r_4593              chloride    0.000000      100.0       True
r_4594                Cu2(+)    0.000000      100.0       True
r_4595                Mn(2+)    0.000000      100.0       True
r_4596                Zn(2+)    0.000000      100.0       True
r_4597                Mg(2+)    0.000000      100.0       True
r_4600                Ca(2+)    0.000000      100.0       True
r_1654              ammonium    0.007698       98.3      False
r_1709            D-fructose    0.270415       41.4      False
r_1714        

In [ ]:
with model:
    meio_de_trabalho(model, q_o2=0.0)   # o nivel de O2 e sobreposto pelo teste
    d9 = T.t9_oxigenio(model, O2, niveis=(0.0, 0.5, 1.0, 2.0, 5.0, 20.0),
                       produtos=PRODUTOS, bio_rid=OBJECTIVO)
print(d9.to_string(index=False))
d9.to_csv(OUT / "T9_oxigenio.csv", index=False)

anaerobio = d9[d9.O2_disponivel == 0.0].iloc[0]
if not anaerobio.cresce:
    achado("T9", "Não cresce em anaerobiose estrita mesmo com esterois e UFA fornecidos", "grave")
else:
    print(f"\ncresce a O2 = 0 com mu = {anaerobio.mu:.4f} h-1")

 O2_disponivel       mu  cresce  estado  ethanol  glycerol  acetate  succinate  acetaldehyde
           0.0 0.000000   False optimal   0.7000    0.0000      0.0        0.0           0.0
           0.5 0.370438    True optimal  31.8303    2.7912      0.0        0.0           0.0
           1.0 0.400712    True optimal  32.0694    2.1011      0.0        0.0           0.0
           2.0 0.461261    True optimal  32.5474    0.7207      0.0        0.0           0.0
           5.0 0.575930    True optimal  30.8788    0.0000      0.0        0.0           0.0
          20.0 0.777525    True optimal   0.0000    0.0000      0.0        0.0           0.0
  [T9] NAO cresce em anaerobiose estrita mesmo com esterois e UFA fornecidos: falta uma via, ou falta um suplemento


## 9.1. T11 - o que bloqueia a anaerobiose

O T9 diz que não cresce sem O₂. O T11 pede ao modelo a produção máxima de cada reagente da biomassa, com e sem oxigénio, e desce recursivamente pelas pseudo-reações dos componentes que aparecem bloqueados. No yeast-GEM o bloqueio está dois
níveis abaixo da biomassa (`r_4041` → `r_4598` → o componente).

In [13]:
d11 = T.t11_bloqueio_anaerobio(model, BIOMASSA, meio_de_trabalho, maxprof=2)
bloq = d11[d11.so_com_O2]
print(d11.to_string(index=False))
d11.to_csv(OUT / "T11_bloqueio_anaerobio.csv", index=False)

if len(bloq):
    achado("T11", "so se fabricam com O2: " +
           ", ".join(f"{r.metabolito} ({r.nome}, via {r.vem_de})"
                     for r in bloq.itertuples()), "grave")

metabolito         nome vem_de  nivel  max_sem_O2  max_com_O2  so_com_O2
    s_0434          ATP r_4041      0    0.841667    1.010000      False
    s_0803          H2O r_4041      0 1000.000000 1000.000000      False
    s_1096        lipid r_4041      0    0.536053    5.714432      False
    s_1212        NADPH r_4041      0    0.050000    0.716667      False
    s_3717      protein r_4041      0    0.895131    0.895131      False
    s_3718 carbohydrate r_4041      0    4.210237    4.991388      False
    s_3719          RNA r_4041      0    6.472443    7.306544      False
    s_3720          DNA r_4041      0   93.518518  113.738738      False
    s_4205     cofactor r_4041      0    0.000000  105.961256       True
    s_4206          ion r_4041      0 1000.000000 1000.000000      False
    s_0529   coenzyme A r_4598      1    0.050000    0.688644      False
    s_0687          FAD r_4598      1    0.561111    0.561111      False
    s_0750  glutathione r_4598      1    1.645486  

## 9.2. T12 - termos residuais da biomassa

Um componente com coeficiente de 10⁻⁶ mmol/gDW não muda a composição de nada e "não contribui" para o crescimento. Mas, pode ser o que impede o modelo de crescer. O T12 retira um de cada vez todos os termos abaixo de 10⁻⁴ e mede o efeito em µ.

A coluna `decide_anaerobiose`: um termo marcado aí está a impor um fenótipo com massa desprezável.

In [14]:
d12 = T.t12_termos_residuais(model, ["r_4041", "r_4598"], meio_de_trabalho,
                             limiar_coef=1e-4)
print(d12.to_string(index=False))
d12.to_csv(OUT / "T12_termos_residuais.csv", index=False)

for r in d12[d12.decide_anaerobiose].itertuples():
    achado("T12", f"{r.metabolito} ({r.nome}) em {r.pseudo}: coef {r.coef:.1e} "
                  f"(~{r.massa_mg_por_gDW:.1e} mg/gDW) decide sozinho o "
                  f"crescimento anaerobio", "grave")

pseudo metabolito        nome          coef  massa_mg_por_gDW  mu_O2_0_com  mu_O2_0_sem  mu_O2_2_com  mu_O2_2_sem  decide_anaerobiose
r_4598     s_0687         FAD -1.000000e-05          0.007825          0.0     0.000000     0.461261     0.461264               False
r_4598     s_0750 glutathione -1.000000e-06          0.000306          0.0     0.000000     0.461261     0.461261               False
r_4598     s_1475         TDP -1.200000e-06          0.000507          0.0     0.000000     0.461261     0.461261               False
r_4598     s_1487         THF -6.340000e-05          0.028112          0.0     0.000000     0.461261     0.461275               False
r_4598     s_3714      heme a -1.000000e-06          0.000852          0.0     0.336697     0.461261     0.461261                True
  [T12] s_3714 (heme a) em r_4598: coef -1.0e-06 (~8.5e-04 mg/gDW) decide sozinho o crescimento anaerobio


## 9.3. T13 - produtos em intervalo, não em ponto

As colunas de secreção do T9 vêm de pFBA, que escolhe entre ótimos alternativos pelo número de passos e não pela bioquímica, logo não serve para afirmar o que o modelo produz. O T13 faz um FVA no ótimo e devolve par (mínimo, máximo):

- mínimo > 0 → o produto é obrigatório naquele ótimo
- mínimo 0 e máximo > 0 → possível mas alternativo, e não se pode afirmar nada
- máximo 0 → o modelo não o consegue produzir

In [15]:
SAIDAS = {k: v for k, v in MAPA["saidas"].items() if v in model.reactions}
with model:
    meio_de_trabalho(model, q_o2=0.0)
    d13 = T.t13_produtos(model, O2, SAIDAS, niveis=(0.0, 0.5, 2.0, 20.0))
print(d13.to_string())
d13.to_csv(OUT / "T13_produtos.csv")

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpwt5j5fco.lp
Reading time = 0.01 seconds
: 2749 rows, 8205 columns, 30901 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmph8tlew2c.lp
Reading time = 0.01 seconds
: 2749 rows, 8205 columns, 30901 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpt8lr_kqt.lp
Reading time = 0.01 seconds
: 2749 rows, 8205 columns, 30901 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use onl

## 9.4. T14 - balanço de carbono e rendimento em etanol

Numa fermentação vínica real, o etanol fica a 90–92 % do máximo teórico de 2 mol por mol de hexose, com o glicerol a consumir 4–6 % do açúcar. O fecho de carbono deve ficar entre 95 e 105 % (?).

In [16]:
linhas = []
for q in (0.0, 2.0, 20.0):
    with model:
        meio_de_trabalho(model, q_o2=q)
        linhas.append({"O2": q, **T.t14_carbono(model, OBJECTIVO,
                                                MAPA["saidas"]["ethanol"])})
d14 = pd.DataFrame(linhas)
print(d14.to_string(index=False))
d14.to_csv(OUT / "T14_carbono.csv", index=False)

  O2       mu  cresce  C_entra_mmol  C_sai_mmol  C_biomassa_mmol  fecho_pct  hexose_consumida  etanol  etanol_por_hexose_mol  pct_do_maximo_teorico trocas_sem_formula
 0.0 0.000000   False        120.40      120.40             0.00      100.0             20.00   23.65                  1.183                   59.1                  -
 2.0 0.461261    True        122.20      103.23            18.91      100.0             20.01   32.55                  1.627                   81.3                  -
20.0 0.777525    True        122.79       90.82            31.88       99.9             20.01   21.97                  1.098                   54.9                  -


## 10. O que apareceu com defeito

Aqui é impressa a lista com os defeitos identificados. A curadoria da secção 11 baseia-se nesta lista. 

In [ ]:
df_ach = pd.DataFrame(ACHADOS)
if len(df_ach):
    print(df_ach.to_string(index=False))
    df_ach.to_csv(OUT / "achados.csv", index=False)
else:
    print("nenhum defeito detetado pelos nove testes.")

teste                                                                                                                                         achado   gravidade
   T6                                                                       3 componentes da biomassa so entram por importacao: mannan, dCMP, heme a       rever
   T8 indispensaveis no meio de trabalho: iron(2+), oxygen, phosphate, potassium, sodium, sulphate, chloride, Cu2(+), Mn(2+), Zn(2+), Mg(2+), Ca(2+) informativo
   T9                                   NAO cresce em anaerobiose estrita mesmo com esterois e UFA fornecidos: falta uma via, ou falta um suplemento       grave
  T11                                                              so se fabricam com O2: s_4205 (cofactor, via r_4041), s_3714 (heme a, via r_4598)       grave
  T12                                              s_3714 (heme a) em r_4598: coef -1.0e-06 (~8.5e-04 mg/gDW) decide sozinho o crescimento anaerobio       grave


## 11. Curadoria

**A anaerobiose é uma condição, não uma correção.** O heme a está certo no modelo aeróbio e só deixa de fazer sentido quando não há O₂ para o sintetizar. Por isso a edição SC1 tem variante = "anaerobia" e aplica-se por função (CU.modo_anaerobio(m, "Sc")), e não por um segundo ficheiro. É também o que o próprio yeast-GEM faz: anaerobicModel.m é uma função aplicada ao modelo consenso e não um yeast-GEM_anaerobic.xml.

In [18]:
import curadoria_minima as CU

disponiveis = CU.edicoes_de("Sc", so_por_omissao=False)
print(pd.DataFrame([{"id": e.id, "variante": e.variante, "titulo": e.titulo[:52],
                     "por_omissao": e.aplicar_por_omissao} for e in disponiveis])
      .to_string(index=False))
for e in disponiveis:
    print(f"\n[{e.id}] {e.titulo}\n   achado : {e.achado}\n   arbitro: {e.arbitro}")

 id  variante                                           titulo  por_omissao
SC1 anaerobia Heme a fora dos cofactores da biomassa anaerobia         True

[SC1] Heme a fora dos cofactores da biomassa anaerobia
   achado : T11 + T12: s_3714 em r_4598 (coef 1e-06) e o unico termo que poe mu = 0 a O2 = 0 com esterois e UFA fornecidos
   arbitro: yeast-GEM, code/otherChanges/anaerobicModel.m, que poe o coeficiente de s_3714 em r_4598 a zero e reequilibra o H+; biossintese do heme em levedura e dependente de O2 em Hem13, Hem14 e Cox15


In [ ]:
EDICOES = {
    "SC1": "T11 + T12 — s_3714 (heme a) em r_4598, coef 1e-06: unico termo que "
           "poe mu = 0 a O2 = 0 com esterois e UFA fornecidos",
}

# Baseline guardado antes da primeira edicao, em aerobiose e em anaerobiose.
from cobra.flux_analysis import pfba

MU_ANTES = {}
for q in (0.0, 2.0):
    with model:
        meio_de_trabalho(model, q_o2=q)
        v = model.slim_optimize()
        MU_ANTES[q] = 0.0 if v != v else v
with model:
    meio_de_trabalho(model, q_o2=2.0)
    PERFIL_ANTES = pfba(model).fluxes.copy()
print(f"baseline: mu(O2=0) = {MU_ANTES[0.0]:.6f} | mu(O2=2) = {MU_ANTES[2.0]:.6f}")

# O modelo curado de base. Para a S. cerevisiae não há nenhuma edicao de
# variante "base": o yeast-GEM nao precisa de nenhuma correção permanente.
curado = model.copy()
curado.id = model.id + "_curado"
base = [eid for eid in EDICOES if CU.CATALOGO[eid].variante in ("base", "ambas")]
LOG_BASE = CU.aplicar(curado, "Sc", base) if base else []
print(f"\nedicoes de variante 'base' aplicadas: {len(LOG_BASE)}")

# A variante anaerobia, em memoria, sobre o curado.
anaerobio = curado.copy()
anaerobio.id = curado.id + "_anaerobio"
LOG_ANA = CU.modo_anaerobio(anaerobio, "Sc", o2_rid=O2, verbose=True)
pd.DataFrame(LOG_BASE + LOG_ANA).to_csv(OUT / "log_edicoes.csv", index=False)

baseline: mu(O2=0) = 0.000000 | mu(O2=2) = 0.461261
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpas27cmy3.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros

edicoes de variante 'base' aplicadas: 0
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp51770dku.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
[SC1] Heme a fora dos cofactores da biomassa anaerobia  (2 operacoes)


### 11.1. O que cada edição mudou

Para cada edição: as reações alteradas com a equação e os bounds (antes e depois).

In [20]:
def diff(antes, depois, titulo):
    print("\n" + "=" * 92); print(titulo); print("=" * 92)
    for r in antes.reactions:
        if r.id not in depois.reactions:
            print(f"  {r.id} (REMOVIDA)"); continue
        novo = depois.reactions.get_by_id(r.id)
        if r.reaction != novo.reaction or r.bounds != novo.bounds:
            print(f"  {r.id}")
            print(f"     antes : {r.reaction}   {r.bounds}")
            print(f"     depois: {novo.reaction}   {novo.bounds}")
    for rid in sorted(set(depois.reactions.list_attr("id")) -
                      set(antes.reactions.list_attr("id"))):
        r = depois.reactions.get_by_id(rid)
        print(f"  {rid} (NOVA): {r.reaction}   {r.bounds}")

diff(model, curado, "variante base (modelo exportado)")
diff(curado, anaerobio, "variante anaerobia (aplicada por CU.modo_anaerobio)")

print("\nlog:")
print(pd.DataFrame(LOG_BASE + LOG_ANA)[["edicao", "op", "reaccao", "nota"]]
      .to_string(index=False))


variante base (modelo exportado)

variante anaerobia (aplicada por CU.modo_anaerobio)
  r_1992
     antes : s_1277 <=>    (-1000.0, 1000.0)
     depois: s_1277 -->    (0.0, 1000.0)
  r_4598
     antes : 0.000190000006114133 s_0529 + 9.99999974737875e-06 s_0687 + 1e-06 s_0750 + 0.0173732001408123 s_0794 + 0.00264999992214143 s_1198 + 0.000150000007124618 s_1203 + 0.000569999974686652 s_1207 + 0.00270000007003546 s_1212 + 0.000989999971352518 s_1405 + 1.20000004244503e-06 s_1475 + 6.34000025456771e-05 s_1487 + 9.99999997475243e-07 s_3714 --> s_4205   (0.0, 1000.0)
     depois: 0.000190000006114133 s_0529 + 9.99999974737875e-06 s_0687 + 1e-06 s_0750 + 0.01737020014081989 s_0794 + 0.00264999992214143 s_1198 + 0.000150000007124618 s_1203 + 0.000569999974686652 s_1207 + 0.00270000007003546 s_1212 + 0.000989999971352518 s_1405 + 1.20000004244503e-06 s_1475 + 6.34000025456771e-05 s_1487 --> s_4205   (0.0, 1000.0)

log:
        edicao      op reaccao                                            

## 12. Repetição de testes sobre o modelo curado

In [ ]:
res = []
for nome, m in (("antes", model), ("curado", curado)):
    for q in (0.0, 2.0, 20.0):
        with m:
            meio_de_trabalho(m, q_o2=q)
            v = m.slim_optimize()
        res.append({"modelo": nome, "O2": q, "mu": round(0.0 if v != v else v, 6)})
with anaerobio:
    meio_de_trabalho(anaerobio, q_o2=0.0)
    v = anaerobio.slim_optimize()
res.append({"modelo": "anaerobio", "O2": 0.0, "mu": round(0.0 if v != v else v, 6)})
print(pd.DataFrame(res).pivot(index="modelo", columns="O2", values="mu").to_string())

print("\nT4 ", T.t4_sistema_fechado(curado, NGAM))
print("T10", T.t10_egc(curado, T.PARES_SC))
print("T10 (anaerobio)", T.t10_egc(anaerobio, T.PARES_SC))

d5b = T.t5_balancos(curado)
print(f"T5  desequilibrio real {len(reais)} -> {len(d5b[~d5b.sem_formula])}")

with anaerobio:
    meio_de_trabalho(anaerobio, q_o2=0.0)
    print("\nT13 anaerobio (min, max):")
    print(T.t13_produtos(anaerobio, O2,
                         {k: v for k, v in MAPA["saidas"].items()
                          if v in anaerobio.reactions}, niveis=(0.0,)).to_string())
    print("\nT14 anaerobio:")
    print(pd.Series(T.t14_carbono(anaerobio, OBJECTIVO,
                                  MAPA["saidas"]["ethanol"])).to_string())

with curado:
    meio_de_trabalho(curado, q_o2=2.0)
    sol_d = pfba(curado)

# Se nenhuma edicao "base" foi aplicada, os dois modelos sao idênticos e não devem existir diferenças
print(f"\nedicoes de variante 'base' aplicadas: {len(LOG_BASE)}"
      + ("  -> as diferencas abaixo sao degenerescencia do pFBA" if not LOG_BASE else ""))
comum = PERFIL_ANTES.index.intersection(sol_d.fluxes.index)
mudou = (sol_d.fluxes[comum] - PERFIL_ANTES[comum]).abs()
mudou = mudou[mudou > 1e-6].sort_values(ascending=False)
print(f"\nreaccoes com fluxo alterado em aerobiose: {len(mudou)} de {len(comum)}")
print(mudou.head(10).to_string())

O2             0.0       2.0       20.0
modelo                                 
anaerobio  0.336697       NaN       NaN
antes      0.000000  0.461261  0.777525
curado     0.000000  0.461261  0.777525

T4  {'biomassa_do_nada': 0.0, 'ATP_do_nada': 0.0}
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpqel74of8.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp4df9za7c.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpxzid541m.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp3eiytfnw.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
T10 {'ATP_c': 0.0, 'ATP_m': 0.0, 'NADH_c': 0.0, 'NADPH_c': 0.0}
Read LP format model from

### 12.1. T15 - rendimento em função do µ e orçamento de carbono

Em anaerobiose a única fonte de ATP é a glicólise, por isso o modelo fermenta tudo o que a estequiometria permitir. Ler o rendimento ao µmax não mede capacidade fermentativa: mede o µmax, que é fixado pela composição da biomassa e pelo GAM.
O µ de uma fermentação vínica anda entre 0.05 e 0.20 h⁻¹ no arranque e cai na fase estacionária.

O orçamento de carbono diz para onde foi, o que não foi para etanol.

In [22]:
ETANOL = MAPA["saidas"]["ethanol"]

print("rendimento em etanol com o mu imposto (O2 = 0, 20 mmol hexose/gDW/h)")
d15 = T.t15_rendimento(anaerobio, OBJECTIVO, ETANOL, meio_de_trabalho)
print(d15.to_string(index=False))
d15.to_csv(OUT / "T15_rendimento.csv", index=False)

resumo, orc = T.t15_orcamento(anaerobio, OBJECTIVO, meio_de_trabalho)
print("\norcamento de carbono ao mu maximo:")
print(pd.Series(resumo).to_string())
print(orc.to_string(index=False))
orc.to_csv(OUT / "T15_orcamento.csv", index=False)

rendimento em etanol com o mu imposto (O2 = 0, 20 mmol hexose/gDW/h)
mu_imposto     mu  hexose  etanol  etanol_por_hexose  pct_do_maximo_teorico  glicerol
       0.0 0.0000    20.0   40.13              2.007                  100.3      0.00
      0.05 0.0500    20.0   39.20              1.960                   98.0      0.00
       0.1 0.1000    20.0   38.26              1.913                   95.6      0.00
       0.2 0.2000    20.0   36.37              1.818                   90.9      0.00
       0.3 0.3000    20.0   34.47              1.723                   86.2      0.03
       max 0.3367    20.0   31.47              1.573                   78.7      3.49

orcamento de carbono ao mu maximo:
mu                 0.3367
C_entra_mmol     121.4700
mmolC_por_gDW     41.1000
       destino  mmol_C  pct
       ethanol   62.94 51.8
carbon dioxide   34.01 28.0
      BIOMASSA   13.84 11.4
      glycerol   10.47  8.6
     succinate    0.14  0.1
     L-glycine    0.06  0.0


## 13. Exportação

A variante anaeróbia não é um segundo ficheiro. Se for preciso nos próximos notebooks -> CU.modo_anaerobio(m, "Sc", o2_rid=...) sobre o ficheiro curado.

In [23]:
XML_OUT = ROOT / "models" / "curados" / "yeast-GEM_curado.xml"
XML_OUT.parent.mkdir(parents=True, exist_ok=True)
cobra.io.write_sbml_model(curado, str(XML_OUT))

relido = cobra.io.read_sbml_model(str(XML_OUT))
with relido:
    meio_de_trabalho(relido, q_o2=2.0)
    mu_aero = relido.slim_optimize()
ana = relido.copy()
CU.modo_anaerobio(ana, "Sc", o2_rid=O2)
with ana:
    meio_de_trabalho(ana, q_o2=0.0)
    mu_ana = ana.slim_optimize()

print(f"{XML_OUT.name}: {len(relido.reactions)} reaccoes")
print(f"   aerobio   (O2=2) mu = {mu_aero:.6f} h-1")
print(f"   anaerobio (O2=0) mu = {mu_ana:.6f} h-1  [CU.modo_anaerobio]")

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpcaayhpvp.lp
Reading time = 0.01 seconds
: 2748 rows, 8204 columns, 30898 nonzeros
yeast-GEM_curado.xml: 4102 reaccoes
   aerobio   (O2=2) mu = 0.461261 h-1
   anaerobio (O2=0) mu = 0.336697 h-1  [CU.modo_anaerobio]


## 14 · Interpretação e limitações

1. **O µ do ficheiro é baixo por meio restrito, não por defeito estrutural.**
   µ = 0.0809 h⁻¹ no meio do ficheiro contra 59.4 com tudo aberto (T2/T3). O número com todas as trocas abertas não é uma taxa de crescimento, é apenas para separar duas hipóteses.
2. **Não há fuga de massa.** T4 dá zero nas duas colunas.
3. **O T6 dá três falsos positivos.** `mannan`, `dCMP` e `heme a` aparecem como "só por importação" mas produzem-se a 11.6, 1.68 e 0.56 mmol/gDW/h. A coluna `so_por_importacao` exclui reações de 2 metabolitos que atravessem compartimentos, por isso marca o que é produzido noutro compartimento e importado. Mede o que só chega por transporte e não o que o modelo não consegue fabricar.
4. **O modelo precisa de O₂, azoto, fósforo, enxofre e os minerais** (T7/T8). Os doze indispensáveis do T8 são todos minerais ou o oxigénio.
5. **Em anaerobiose estrita não cresce** T11 localiza `s_4205` (cofactor) e, um nível abaixo, `s_3714` (heme a). T12 mostra
   que esse termo pesa ~8.5×10⁻⁴ mg/gDW e ainda assim decide o crescimento. Com a SC1 aplicada: µ = 0.337 h⁻¹, etanol 31.5, glicerol 3.5, fecho de carbono 100.0 %, rendimento 1.57 mol etanol/mol hexose (79 % do máximo teórico). O µ aeróbio não se altera.